# Reproduce: prenatal gene-phenotype annotation and gene ranking with the 2020 and 2026 HPO releases

Run All (Kernel > Restart Kernel and Run All Cells) downloads the public inputs, runs every analysis step in order, redraws the figures and compares every regenerated file in `outputs/` with the committed results in `reference/`. The last cell prints `REPRODUCED` when all files agree. A full run takes a few minutes.

In [ ]:
import os, sys, json, glob, math, shutil, subprocess
import pandas as pd
from IPython.display import Image, display

# outputs/ is rebuilt from scratch, the committed results in reference/ are only read
shutil.rmtree('outputs', ignore_errors=True)
os.makedirs('outputs')

def run(script):
    """Runs one pipeline step from scripts/, prints its output and stops with the full error message on failure."""
    cmd = ['bash', script] if script.endswith('.sh') else [sys.executable, script]
    r = subprocess.run(cmd, cwd='scripts', capture_output=True, text=True)
    print(f'--- {script}: exit {r.returncode}')
    if r.stderr.strip() and not r.returncode:
        print(r.stderr[-5000:])
    if r.returncode:
        print(r.stdout[-5000:]); print(r.stderr[-5000:])
        raise RuntimeError(f'{script} failed')


## 1. Public inputs

Downloads the HPO releases v2026-09-01 and 2020-10-12, the HGNC complete set 2026-09-04 and the supplementary table of the primary cohort (Fu et al. 2022) into `inputs/`. A failed download or a file that differs from `data/SHA256SUMS` is replaced by the archived copy in `data/inputs.tar.xz`.

In [ ]:
run('download.sh')
print(open('inputs/RELEASE.txt').read())

## 2. Cases and gene identifiers

Reads the findings and genes of the 229 primary-cohort fetuses from the source table and of the 106 further fetuses from `inputs_manual/external_cases.csv`, maps findings to HPO terms and harmonises gene symbols to current HGNC symbols through NCBI Gene IDs.

In [ ]:
run('fu_cases.py')
run('gene_audit.py')
A = json.load(open('outputs/gene_audit.json'))
{k: A[k] for k in ['causal_genes', 'fetuses', 'renamed_in_sources', 'universe_by_symbol', 'universe_harmonised', 'harmonised']}

## 3. Ranking with the 2020 and 2026 releases (primary outcome)

Ranks all genes annotated in both releases for every fetus by the Resnik best-match average with identical queries. Primary outcome: causal gene in the top 10, primary cohort, McNemar exact test.

In [ ]:
run('rank.py')
R = json.load(open('outputs/rank_summary.json'))
pd.DataFrame({c: {k: v for k, v in R[c].items() if k.startswith(('fetuses', 'top10_2020', 'top10_2026', 'median_rank_2020', 'median_rank_2026', 'mcnemar', 'wilcoxon', 'improved', 'worsened'))}
              for c in ['fu2022', 'further', 'graham2026', 'smogavec2022', 'aogs2026']}).T

## 4. Secondary outcomes and leakage check

Annotation of each finding to the causal gene, fetuses without an annotated term in the organ system of a finding, top-10 counts by mode of inheritance, and annotations citing the primary cohort publication.

In [ ]:
run('gaps.py')
G = json.load(open('outputs/gaps.json'))
{k: G[k] for k in ['fetus_none_exact_2026', 'by_inheritance', 'leakage']}

## 5. Agreement of the two independent extractions and sensitivity analysis

Compares the two independent extractions of the further cohorts and repeats the comparison without the 12 fetuses whose diagnosis was uncertain in the source.

In [ ]:
run('agreement.py')
run('sensitivity.py')
display(json.load(open('outputs/agreement.json')))
json.load(open('outputs/sensitivity.json'))

## 6. Exploratory analyses X1 to X4 and descriptive analyses P1 to P5

X3 supplements the annotation of the causal gene with the findings of the other fetuses with the same gene (leave one fetus out). P2 describes the prenatal branch of the 2026 release, P4 lists candidate annotations (Table S1).

In [ ]:
run('explore.py')
run('paper_analyses.py')
X = json.load(open('outputs/explore.json'))
{k: X[k] for k in ['x1', 'x3', 'x4']}

## 7. Robustness analyses R1 to R7

Prenatal re-coding, alternative scores, each release's own gene set, mechanism of the lower 2026 ranks, cross-cohort supplementation, permutation control (1,000 permutations), supplementation of all genes of the primary cohort, dual coding and a worked example. Also checks that the matrix scorer reproduces every rank.

In [ ]:
run('robustness.py')
run('robustness_r6.py')
W = json.load(open('outputs/robustness.json'))
{k: W[k] for k in ['check_max_abs_diff', 'r2', 'r5b', 'r5c']}

## 8. Paired differences, annotation groups and fetal term linkage

Paired differences in top-k proportions with Tango score 95% confidence intervals (the interval function is first checked against reference values from R PropCIs), the prenatal branch by clinical group, and fetal terms without an organ-system ancestor.

In [ ]:
run('paired_ci.py')
run('paired_differences.py')
run('annotation_groups.py')
display(json.load(open('outputs/annotation_groups.json')))
pd.read_csv('outputs/paired_ci.tsv', sep='\t')

## 9. Figures

Figure 1 (study flow), Figures 2 to 4 and Figure S1 (worked example of the ranking).

In [ ]:
run('flowchart.py')
run('paper_figures.py')
run('s1_figure.py')
for f in ['study_flow/study_flow.png'] + [f'paper/figure{k}.png' for k in (2, 3, 4)] + ['s1/figure_s1.png']:
    display(Image(os.path.join('figures', f), width=800))

## 10. Comparison with the committed results

In [ ]:
# every file in reference/ must be regenerated in outputs/ with the same content (numbers to a relative 1e-9)
def same(a, b):
    if isinstance(a, dict):
        return isinstance(b, dict) and a.keys() == b.keys() and all(same(a[k], b[k]) for k in a)
    if isinstance(a, list):
        return isinstance(b, list) and len(a) == len(b) and all(same(x, y) for x, y in zip(a, b))
    if isinstance(a, float) or isinstance(b, float):
        return isinstance(a, (int, float)) and isinstance(b, (int, float)) and math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-12)
    return a == b

def same_table(f, g):
    a, b = pd.read_csv(f, sep='\t', dtype=str), pd.read_csv(g, sep='\t', dtype=str)
    if a.shape != b.shape or list(a.columns) != list(b.columns):
        return False
    for c in a.columns:
        x, y = pd.to_numeric(a[c], errors='coerce'), pd.to_numeric(b[c], errors='coerce')
        num = x.notna() & y.notna()
        if not ((x[num] - y[num]).abs() <= 1e-9 * (1 + y[num].abs())).all():
            return False
        if not a[c][~num].fillna('').equals(b[c][~num].fillna('')):
            return False
    return True

ref = sorted(os.listdir('reference'))
new = sorted(os.listdir('outputs'))
missing, extra = sorted(set(ref) - set(new)), sorted(set(new) - set(ref))
different = [f for f in ref if f in new and not (
    same(json.load(open(f'reference/{f}')), json.load(open(f'outputs/{f}'))) if f.endswith('.json')
    else same_table(f'reference/{f}', f'outputs/{f}'))]
print(f'{len(ref)} committed result files, {len(new)} regenerated')
print('missing:', missing, 'extra:', extra, 'different:', different)
print('REPRODUCED' if not (missing or extra or different) else 'NOT REPRODUCED')
